In [1]:
!pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq


## Load the raw data

In [3]:

df = pd.read_parquet(r'C:\Users\harsh\OneDrive\Desktop\data Engineering\Midterm-Project\data\raw.parquet')

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
df.shape

(3475226, 20)

In [5]:
#check missing values
df.isnull().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64

In [6]:
#Convert Datetime Columns
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

# Drop rows with invalid timestamps
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])

print("After datetime cleaning:", df.shape)

After datetime cleaning: (3475226, 20)


In [7]:
#Create trip_duration in minutes
df["trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df[["trip_duration"]].describe()

,trip_duration
count,3.475226e+06
mean,1.501812e+01
std,3.871358e+01
min,-5.147232e+04
25%,7.283333e+00
50%,1.170000e+01
75%,1.833333e+01
max,5.626317e+03


In [8]:
# Remove invalid values
df = df[df["trip_distance"] > 0]
df = df[df["fare_amount"] > 0]
df = df[df["trip_duration"] > 0]
df = df[df["passenger_count"] > 0]

print("After removing invalid trips:", df.shape)

After removing invalid trips: (2815614, 21)


In [9]:
#remove extra outliers
df = df[df["trip_distance"] < 100]
df = df[df["trip_duration"] < 300]   # < 5 hours
df = df[df["fare_amount"] < 500]

print("After removing outliers:", df.shape)

After removing outliers: (2814366, 21)


## Handle missing values

In [10]:
# Handle missing values in categorical columns
numeric_cols = [
    "tip_amount",
    "tolls_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df.isnull().sum()

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
trip_duration            0
dtype: int64

## FEATURE ENGINEERING

In [11]:

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["is_weekend"] = df["pickup_day"].isin([5,6]).astype(int)
df["is_morning_peak"] = df["pickup_hour"].between(7, 9).astype(int)
df["is_evening_peak"] = df["pickup_hour"].between(16, 19).astype(int)
df[["pickup_hour", "pickup_day", "is_weekend"]].head()

,pickup_hour,pickup_day,is_weekend
0,0,2,0
1,0,2,0
2,0,2,0
3,0,2,0
4,0,2,0


In [13]:
df.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,...,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month,is_weekend,is_morning_peak,is_evening_peak
count,2.814366e+06,2814366,2814366,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,...,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06,2.814366e+06
mean,1.776679e+00,2025-01-17 02:34:11.195629,2025-01-17 02:48:47.221112,1.306534e+00,3.176962e+00,2.422437e+00,1.670561e+02,1.661377e+02,1.175405e+00,1.793597e+01,...,2.325495e+00,1.329645e-01,4.873375e-01,1.460042e+01,1.438705e+01,3.018545e+00,1.000082e+00,2.474458e-01,1.072796e-01,2.763063e-01
min,1.000000e+00,2024-12-31 20:47:55,2024-12-31 20:54:00,1.000000e+00,1.000000e-02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e-02,...,0.000000e+00,0.000000e+00,0.000000e+00,1.666667e-02,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000e+00,2025-01-09 16:23:35.250000,2025-01-09 16:40:12.500000,1.000000e+00,9.900000e-01,1.000000e+00,1.320000e+02,1.160000e+02,1.000000e+00,8.600000e+00,...,2.500000e+00,0.000000e+00,0.000000e+00,7.066667e+00,1.100000e+01,2.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+00,2025-01-16 21:12:27,2025-01-16 21:25:51,1.000000e+00,1.630000e+00,1.000000e+00,1.620000e+02,1.630000e+02,1.000000e+00,1.210000e+01,...,2.500000e+00,0.000000e+00,7.500000e-01,1.135000e+01,1.500000e+01,3.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.000000e+00,2025-01-24 17:36:18,2025-01-24 17:52:04,1.000000e+00,3.000000e+00,1.000000e+00,2.340000e+02,2.360000e+02,1.000000e+00,1.910000e+01,...,2.500000e+00,0.000000e+00,7.500000e-01,1.813333e+01,1.900000e+01,4.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
max,2.000000e+00,2025-02-01 00:00:44,2025-02-01 01:10:26,9.000000e+00,9.758000e+01,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,4.830000e+02,...,2.500000e+00,1.750000e+00,7.500000e-01,2.965333e+02,2.300000e+01,6.000000e+00,1.200000e+01,1.000000e+00,1.000000e+00,1.000000e+00
std,4.164718e-01,NaN,NaN,7.420581e-01,4.300039e+00,1.145499e+01,6.294697e+01,6.885292e+01,4.781003e-01,1.689842e+01,...,6.370321e-01,4.636897e-01,3.577783e-01,1.179944e+01,5.526593e+00,1.837375e+00,3.005356e-02,4.315280e-01,3.094684e-01,4.471702e-01


## Save cleaned data

In [12]:

df.to_parquet(
    r'C:\Users\harsh\OneDrive\Desktop\data Engineering\Midterm-Project\data\cleaned.parquet',
    index=False
)

print("✅ Cleaned data saved successfully!")

✅ Cleaned data saved successfully!
